# Data Filter

Athlete Pose Dataset

[explain the dataset]
21 GB Pose data

Find annotations COCO file and extract only these files at first

In [ ]:
import zipfile
import os

# run this in same dir of pose_2d.zip
zip_file_path = 'pose_2d.zip' 

# uncomment target_folder to run successfully
# target_folder = 'pose_2d/annotations/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()
    
    json_files_to_extract = [
        f for f in all_files 
        if target_folder in f and f.endswith('.json')
    ]
    
    if not json_files_to_extract:
        print("Could not find the annotations folder.")
    else:
        print(f"Found {len(json_files_to_extract)} JSON files. Extracting now...")
        
        for file in json_files_to_extract:
            print(f" -> Extracting: {file}")
            zip_ref.extract(file)

In [1]:
import json

file_path = 'datasets/athlete_pose/pose_2d/annotations/train_set.json'
print(f"Loading {file_path}...")

with open(file_path, 'r') as f:
    coco_data = json.load(f)

print("\n--- Keys in the main COCO dictionary ---")
print(coco_data.keys())

print("\n--- First Image Record ---")
print(json.dumps(coco_data['images'][0], indent=4))

Loading datasets/athlete_pose/pose_2d/annotations/train_set.json...

--- Keys in the main COCO dictionary ---
dict_keys(['images', 'annotations', 'categories', 'info'])

--- First Image Record ---
{
    "id": 20000000001,
    "width": 1280,
    "height": 768,
    "file_name": "20000000001.jpg"
}


Looking at the annotations .json files, it was not clear which images were running. I decided to look at a random selection of images from the training set to determine if there was a number range that all the running images were in.

In [ ]:
import random
import os
import zipfile

# run in Downloads/ next to pose_2d.zip
# zip_file_path = 'pose_2d.zip' 
# image_folder_in_zip = 'pose_2d/test_set/'
# output_dir = '../Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/train_set/manual_curation/'

os.makedirs(output_dir, exist_ok=True)

train_set_filenames = []

print("Reading zip index...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()
    
    # Find all JPGs in the train_set folder
    image_files = [f for f in all_files if f.startswith(image_folder_in_zip) and f.endswith('.jpg')]
    
    print(f"Found {len(image_files)} total images.")
    
    # Randomly select 1,000 images
    sample_size = min(1000, len(image_files)) 
    sampled_files = random.sample(image_files, sample_size)
    
    print(f"Extracting {sample_size} random images to {output_dir}...")
    for i, file in enumerate(sampled_files):
        # Extract just the filename (e.g., '20000000001.jpg') without the long folder path
        filename = os.path.basename(file)
        train_set_filenames.append(filename)
        
        # Read the image bytes from the zip and write them to our new folder
        source = zip_ref.open(file)
        target = open(os.path.join(output_dir, filename), "wb")
        with source, target:
            target.write(source.read())
            
        if i % 100 == 0 and i > 0:
            print(f"  ...extracted {i} images")

Found that running images were in:

training set: files 200*.jpeg and 300*.jpeg

test_set: 200*.jpeg

validation_set: 200*.jpeg

In [ ]:
import zipfile
import os

# run next to pose_2d.zip
# zip_file_path = 'pose_2d.zip'
# image_folder_in_zip = 'pose_2d/valid_set/'
# output_dir = '../Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/valid_set/'

TARGET_PREFIXES = ('2000') 

os.makedirs(output_dir, exist_ok=True)

print(f"Reading zip index from {zip_file_path}...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()
    
    running_files = [
        f for f in all_files 
        if f.startswith(image_folder_in_zip) 
        and f.endswith('.jpg') 
        and os.path.basename(f).startswith(TARGET_PREFIXES)
    ]
    
    print(f"Found {len(running_files)} running images! Extracting now...")
    
    for i, file in enumerate(running_files):
        filename = os.path.basename(file)
        
        source = zip_ref.open(file)
        target = open(os.path.join(output_dir, filename), "wb")
        with source, target:
            target.write(source.read())
            
        if (i + 1) % 500 == 0:
            print(f"  ...extracted {i + 1} images")

print(f"{len(running_files)} running images saved to {output_dir}")

use this script to extract the running images and make training, test, and validation sets

now filter the COCO file to relevant running images

In [ ]:
import json
import os

# input_json_path = 'datasets/athlete_pose/pose_2d/annotations/train_set.json'
# output_json_path = 'datasets/athlete_pose/pose_2d/annotations/train_set_running_only.json'

TARGET_PREFIXES = ('2000', '3000')

os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

print(f"Loading original annotations from {input_json_path}...")
with open(input_json_path, 'r') as f:
    coco_data = json.load(f)

filtered_images = []
kept_image_ids = set()

for img in coco_data.get('images', []):
    filename = img.get('file_name', '')
    if filename.startswith(TARGET_PREFIXES):
        filtered_images.append(img)
        kept_image_ids.add(img['id'])

filtered_annotations = [
    ann for ann in coco_data.get('annotations', [])
    if ann.get('image_id') in kept_image_ids
]

new_coco_data = {
    "images": filtered_images,
    "annotations": filtered_annotations,
    "info": coco_data.get("info", {}),
    "licenses": coco_data.get("licenses", []),
    "categories": coco_data.get("categories", [])
}

print(f"Saving filtered data to {output_json_path}...")
with open(output_json_path, 'w') as f:
    json.dump(new_coco_data, f)

print("\n--- Filter Summary ---")
print(f"Images kept: {len(filtered_images)} of {len(coco_data.get('images', []))}")
print(f"Annotations kept: {len(filtered_annotations)} of {len(coco_data.get('annotations', []))}")

Loading original annotations from datasets/athlete_pose/pose_2d/annotations/train_set.json...
Saving filtered data to datasets/athlete_pose/pose_2d/annotations/train_set_running_only.json...

--- Filter Summary ---
Images kept: 10504 of 71375
Annotations kept: 10504 of 71375


Convert COCO to YOLO format

[explain how this works]

In [5]:
from ultralytics.data.converter import convert_coco

convert_coco(labels_dir='datasets/athlete_pose/pose_2d/annotations/', use_keypoints=True)

Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/test_set.json: 100% ━━━━━━━━━━━━ 28402/28402 8.9Kit/s 3.2s0.0s
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/test_set_running_only.json: 100% ━━━━━━━━━━━━ 2936/2936 10.6Kit/s 0.3s.2s
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/train_set.json: 100% ━━━━━━━━━━━━ 71375/71375 8.6Kit/s 8.3s0.1ss
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/train_set_running_only.json: 100% ━━━━━━━━━━━━ 10504/10504 9.8Kit/s 1.1s.1s
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/valid_set.json: 100% ━━━━━━━━━━━━ 29789/29789 9.6Kit/s 3.1s<0.0s
Annotations /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/athlete_pose/pose_2d/annotations/valid_set_runni

Now the data is ready for fine-tuning the YOLO model. The converted data has the following format:

/dataset
├── train/
│   ├── images/
│   └── labels/
├── test/
│   ├── images/
│   └── labels/
└── valid/
    ├── images/
    └── labels/